A estas alturas, esperamos haberte convencido de que groupby() es una herramienta excelente para agrupar datos y realizar análisis más complejos. Aunque siempre puedes usar groupby() para calcular las propiedades agregadas de tus datos, pandas también ofrece tablas dinámicas como método alternativo para agrupar y analizar datos.

Las tablas dinámicas son una gran herramienta para sintetizar conjuntos de datos y explorar sus diferentes dimensiones. Son muy populares en las aplicaciones de hojas de cálculo como Excel, pero es aún más impresionante crearlas mediante programación con pandas.

Echemos otro vistazo a los datos de ventas de videojuegos a continuación:

```python

import pandas as pd

df = pd.read_csv('/datasets/vg_sales.csv')
print(df.head())
```


Supongamos que queremos determinar el total de ventas en Europa de cada género en cada plataforma. Las tablas dinámicas ofrecen un método rápido y cómodo para conseguirlo. Primero, vamos a examinar el código y luego a comentarlo. Para simplificar las cosas, eliminaremos las filas con valores ausentes.

```python
df = pd.read_csv('/datasets/vg_sales.csv')
df.dropna(inplace=True)

pivot_data = df.pivot_table(index='genre',
                            columns='platform',
                            values='eu_sales',
                            aggfunc='sum'
                           )
print(pivot_data)
print()
print(type(pivot_data))
```

Creamos una tabla dinámica utilizando el método pivot_table(), que tiene un nombre muy apropiado. Los parámetros que utilizamos fueron:

- index=: la columna cuyos valores se convierten en índices en la tabla dinámica;
- columns=: la columna cuyos valores se convierten en columnas en la tabla dinámica;
- values=: la columna cuyos valores queremos agregar en la tabla dinámica;
- aggfunc=: la función de agregación que queremos aplicar a los valores en cada grupo de filas y columnas.

Cada celda de la tabla dinámica de arriba contiene las ventas totales en Europa para cada combinación particular de género/plataforma. También imprimimos el tipo de datos de la tabla dinámica para mostrar que es un DataFrame de pandas, que ya conocemos muy bien.

El uso de una tabla dinámica aquí es conveniente porque nos permite fácilmente excluir todas las columnas de df que no nos interesan para nuestro análisis. Además, puede ser más fácil de leer que el resultado equivalente de groupby(), como puedes ver a continuación.

```python
df = pd.read_csv('/datasets/vg_sales.csv')
df.dropna(inplace=True)

groupby_data = df.groupby(['genre', 'platform'])['eu_sales'].mean()
print(groupby_data)
print()
print(type(groupby_data))
```

Un formato muy diferente, ¿verdad? Observa también que el resultado de groupby() devuelve un objeto Series, mientras que pivot_table() devuelve un DataFrame. Ya sea que elijas usar groupby() o pivot_table() depende de tus preferencias personales y, con el tiempo, desarrollarás la intuición sobre qué herramienta es la más conveniente para la tarea en cuestión.

### En esta lección volveremos al conjunto de datos de ventas de videojuegos. Aquí están las primeras filas para recordarte su estructura:

```python
df = pd.read_csv('/datasets/vg_sales.csv')
print(df.head())
```

                       name platform  year_of_release         genre publisher  \
0                Wii Sports      Wii           2006.0        Sports  Nintendo   
1         Super Mario Bros.      NES           1985.0      Platform  Nintendo   
2            Mario Kart Wii      Wii           2008.0        Racing  Nintendo   
3         Wii Sports Resort      Wii           2009.0        Sports  Nintendo   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing  Nintendo   

  developer  na_sales  eu_sales  jp_sales  critic_score  user_score  
0  Nintendo     41.36     28.96      3.77          76.0         8.0  
1       NaN     29.08      3.58      6.81           NaN         NaN  
2  Nintendo     15.68     12.76      3.79          82.0         8.3  
3  Nintendo     15.61     10.93      3.28          80.0         8.0  
4       NaN     11.27      8.89     10.22           NaN         NaN

Queremos conocer algunas estadísticas generales de las distribuidoras de juegos, en particular:

su puntuación promedio de la crítica;
sus ventas totales.
Como ya hemos visto, podemos hacerlo usando groupby(). Primero, obtengamos la puntuación de reseña promedio para cada distribuidora:

```python
df = pd.read_csv('/datasets/vg_sales.csv')

mean_score = df.groupby('publisher')['critic_score'].mean()
print(mean_score)
```


publisher
10TACLE Studios                 42.000000
1C Company                      73.000000
20th Century Fox Video Games          NaN
2D Boy                          90.000000
3DO                             57.470588
                                  ...    
id Software                     85.000000
imageepoch Inc.                       NaN
inXile Entertainment            81.000000
mixi, Inc                             NaN
responDESIGN                          NaN
Name: critic_score, Length: 581, dtype: float64


Obtengamos también el número de ventas. La forma más sencilla de hacerlo es con un segundo groupby():

```python

df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales']
num_sales = df.groupby('publisher')['total_sales'].sum()
print(num_sales)
```

publisher
10TACLE Studios                 0.11
1C Company                      0.08
20th Century Fox Video Games    1.92
2D Boy                          0.03
3DO                             9.52
                                ... 
id Software                     0.02
imageepoch Inc.                 0.04
inXile Entertainment            0.09
mixi, Inc                       0.87
responDESIGN                    0.13
Name: total_sales, Length: 581, dtype: float64


Observa que el índice de ambos resultados es la columna 'publisher' porque agrupamos por 'publisher' en ambos casos. Como ambos resultados comparten el mismo índice, podemos unir fácilmente los resultados en un DataFrame usando la función concat() de pandas:

```python

df = pd.read_csv('/datasets/vg_sales.csv')

mean_score = df.groupby('publisher')['critic_score'].mean()

df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales']
num_sales = df.groupby('publisher')['total_sales'].sum()

df_concat = pd.concat([mean_score, num_sales], axis='columns')
print(df_concat)
```

                              critic_score  total_sales
publisher                                              
10TACLE Studios                  42.000000         0.11
1C Company                       73.000000         0.08
20th Century Fox Video Games           NaN         1.92
2D Boy                           90.000000         0.03
3DO                              57.470588         9.52
...                                    ...          ...
id Software                      85.000000         0.02
imageepoch Inc.                        NaN         0.04
inXile Entertainment             81.000000         0.09
mixi, Inc                              NaN         0.87
responDESIGN                           NaN         0.13


En general, concat() espera una lista de objetos de tipo Series y/o DataFrame. Para obtener nuestro resultado, pasamos una lista de variables de Series a concat() y configuramos axis='columns' para asegurarnos de que se combinaran como columnas.

Ten en cuenta que los nombres de las columnas originales se conservan en el DataFrame concatenado.

Podemos renombrar columnas utilizando el método columns. Se puede llamar en un DataFrame y pasarle una lista de nuevos nombres de columnas para reemplazar las existentes. Los nuevos nombres deben pasarse en el mismo orden que los nombres originales de las columnas.

Cambiemos el nombre de 'critic_score', ya que ahora representa un promedio:

```python

df = pd.read_csv('/datasets/vg_sales.csv')

mean_score = df.groupby('publisher')['critic_score'].mean()

df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales']
num_sales = df.groupby('publisher')['total_sales'].sum()

df_concat = pd.concat([mean_score, num_sales], axis='columns')
df_concat.columns = ['avg_critic_score', 'total_sales']
print(df_concat)
```
                              avg_critic_score  total_sales
publisher                                                  
10TACLE Studios                      42.000000         0.11
1C Company                           73.000000         0.08
20th Century Fox Video Games               NaN         1.92
2D Boy                               90.000000         0.03
3DO                                  57.470588         9.52
...                                        ...          ...
id Software                          85.000000         0.02
imageepoch Inc.                            NaN         0.04
inXile Entertainment                 81.000000         0.09
mixi, Inc                                  NaN         0.87
responDESIGN                               NaN         0.13

En general, es una buena idea cambiar el nombre de las columnas después de agruparlas y procesarlas para que sean más indicativas de cómo se procesaron las columnas.

Es posible que hayas notado que podríamos obtener el mismo resultado que antes usando agg(). Sin embargo, concat() es bastante versátil. Podemos utilizarlo para concatenar DataFrames:

por filas, suponiendo que tengan el mismo número de columnas;
por columnas si tienen el mismo número de filas.
Para concatenar filas de DataFrames separados, podemos usar concat() y configurar axis='index' (u omitir este parámetro, ya que axis='index' es el argumento predeterminado). Alternativamente, podemos utilizar números enteros para el argumento index=, donde index=0 concatenará filas y index=1 concatenará columnas.

Aquí hay un ejemplo en el que filtramos los datos en dos DataFrames separados según el género y luego los recombinamos en un solo DataFrame:

```python

df = pd.read_csv('/datasets/vg_sales.csv')

rpgs = df[df['genre'] == 'Role-Playing']
platformers = df[df['genre'] == 'Platform']

df_concat = pd.concat([rpgs, platformers])
print(df_concat[['name', 'genre']])

```

                                                   name         genre
4                              Pokemon Red/Pokemon Blue  Role-Playing
12                          Pokemon Gold/Pokemon Silver  Role-Playing
20                        Pokemon Diamond/Pokemon Pearl  Role-Playing
25                        Pokemon Ruby/Pokemon Sapphire  Role-Playing
27                          Pokemon Black/Pokemon White  Role-Playing
...                                                 ...           ...
16356                                    Strider (2014)      Platform
16358                                Goku Makaimura Kai      Platform
16603  The Land Before Time: Into the Mysterious Beyond      Platform
16710                Woody Woodpecker in Crazy Castle 5      Platform
16715                                  Spirits & Spells      Platform

Acabas de aprender a combinar DataFrames concatenándolos por filas o columnas usando concat().

La concatenación de DataFrames conserva la cantidad total de datos. Por ejemplo, combinar un DataFrame que tiene dos columnas y tres filas con otro DataFrame que tiene las mismas dos columnas y cinco filas da como resultado un DataFrame con dos columnas y ocho filas. El número total de celdas antes y después de la concatenación es dieciséis.

En esta lección, aprenderás a combinar DataFrames utilizando el método merge() de forma que afecte a la cantidad de datos con los que estás trabajando.

Considera el siguiente ejemplo: dos estudiantes de literatura acuerdan que uno escribirá la mitad de la lista de lectura de verano de la pizarra mientras el otro mira YouTube. Después el primero irá a la cafetería, mientras que el segundo copia el resto de la lista. Finalmente, los dos combinarán las listas. ¡Trabajo en equipo! Vamos a ver cómo les fue:

```python

first_pupil_df = pd.DataFrame(
    {
        'author': ['Alcott', 'Fitzgerald', 'Steinbeck', 'Twain', 'Hemingway'],
        'title': ['Little Women',
                  'The Great Gatsby',
                  'Of Mice and Men',
                  'The Adventures of Tom Sawyer',
                  'The Old Man and the Sea'
                 ],
    }
)
second_pupil_df = pd.DataFrame(
    {
        'author': ['Steinbeck', 'Twain', 'Hemingway', 'Salinger', 'Hawthorne'],
        'title': ['East of Eden',
                  'The Adventures of Huckleberry Finn',
                  'For Whom the Bell Tolls',
                  'The Catcher in the Rye',
                  'The Scarlett Letter'
                 ],
    }
)

print(first_pupil_df)
print()
print(second_pupil_df)
```

       author                         title
0      Alcott                  Little Women
1  Fitzgerald              The Great Gatsby
2   Steinbeck               Of Mice and Men
3       Twain  The Adventures of Tom Sawyer
4   Hemingway       The Old Man and the Sea

      author                               title
0  Steinbeck                        East of Eden
1      Twain  The Adventures of Huckleberry Finn
2  Hemingway             For Whom the Bell Tolls
3   Salinger              The Catcher in the Rye
4  Hawthorne                 The Scarlett Letter



# Unión interna
Usemos el método merge() para combinar entradas que tienen los mismos autores. El nombre de la columna en la que se realizará la fusión se pasa al parámetro on=, en este caso, 'author':

```python

both_pupils = first_pupil_df.merge(second_pupil_df, on='author')
print(both_pupils) 
```
      author                       title_x                             title_y
0  Steinbeck               Of Mice and Men                        East of Eden
1      Twain  The Adventures of Tom Sawyer  The Adventures of Huckleberry Finn
2  Hemingway       The Old Man and the Sea             For Whom the Bell Tolls



El resultado contiene solo aquellos autores que están presentes en ambos DataFrames originales.

El DataFrame fusionado incluye todas las columnas de los DataFrames originales, pero solo se conservan las filas con autores compartidos. Ya que ambos DataFrames originales tienen una columna llamada 'title', pandas agregó los sufijos _x y _y para diferenciarlas en el DataFrame fusionado. Cabe destacar que el DataFrame fusionado solo tiene 9 celdas, frente a las 20 celdas de los DataFrame originales: ¡la cantidad de datos ha cambiado!

Este modo de fusionar se denomina unión interna (inner merge). Existen otros tipos de fusiones, que pueden especificarse con el parámetro how= de merge(). Pero 'inner' es el argumento por defecto para how=, así que no necesitamos incluirlo arriba.

# Unión externa
Una unión externa (outer merge) se diferencia de una unión interna en que todos los valores en la columna especificada se conservan de ambos DataFrames originales, pero el DataFrame fusionado tiene valores ausentes donde no hay ninguna coincidencia. Lo mejor es ilustrar esto con un ejemplo:

```py
both_pupils = first_pupil_df.merge(second_pupil_df, on='author', how='outer')
print(both_pupils)
```

       author                       title_x  \
0      Alcott                  Little Women   
1  Fitzgerald              The Great Gatsby   
2   Steinbeck               Of Mice and Men   
3       Twain  The Adventures of Tom Sawyer   
4   Hemingway       The Old Man and the Sea   
5    Salinger                           NaN   
6   Hawthorne                           NaN   

                              title_y  
0                                 NaN  
1                                 NaN  
2                        East of Eden  
3  The Adventures of Huckleberry Finn  
4             For Whom the Bell Tolls  
5              The Catcher in the Rye  
6                 The Scarlett Letter

Hay 7 autores únicos en ambos DataFrames originales, cada uno se representa con una fila en el DataFrame fusionado. Para los autores en el primer DataFrame que no están también en el segundo (es decir, 'Alcott' y 'Fitzgerald'), hay valores NaN en la columna que proviene del segundo DataFrame (es decir, 'title_y'), y viceversa. Además, observa que ahora tenemos 21 celdas de datos.

# Unión izquierda
El último tipo de unión que nos gustaría discutir es la unión izquierda (left merge), que podemos realizar pasando how='left' a merge(). En una unión izquierda, todos los valores del DataFrame izquierdo (en el que llamamos merge()) están presentes en el DataFrame fusionado. Los valores del DataFrame derecho (el que pasamos como entrada a merge()) solo se conservan para los valores que coinciden con la columna especificada en el DataFrame izquierdo. Una vez más, se explica mejor con un ejemplo:

```py
both_pupils = first_pupil_df.merge(second_pupil_df, on='author', how='left')
print(both_pupils)
```

       author                       title_x  \
0      Alcott                  Little Women   
1  Fitzgerald              The Great Gatsby   
2   Steinbeck               Of Mice and Men   
3       Twain  The Adventures of Tom Sawyer   
4   Hemingway       The Old Man and the Sea   

                              title_y  
0                                 NaN  
1                                 NaN  
2                        East of Eden  
3  The Adventures of Huckleberry Finn  
4             For Whom the Bell Tolls



Como puedes ver, todos los autores y títulos del primer estudiante están en el DataFrame fusionado, pero las filas con 'Salinger' y 'Hawthorne' del segundo estudiante no lo están porque esos autores no aparecen en el DataFrame del primer estudiante.

Esta unión izquierda contiene 15 celdas de datos, que difieren de la cantidad original y las cantidades de cada una de las otras uniones que hicimos.

Ten en cuenta que también existe una unión derecha (right merge, how='right'). Sin embargo, funciona de manera idéntica a una unión izquierda, excepto que el DataFrame combinado conserva todos los valores del DataFrame derecho en vez del izquierdo. Se puede lograr el mismo resultado realizando una unión izquierda y cambiando el orden de los DataFrames.

Aquí tienes un diagrama de Venn que ilustra todas las opciones de fusión que hemos comentado, para que sea aún más fácil de entender: (esta en la carpeta)

# Tener en cuenta los nombres de las columnas
Hay dos aspectos de todas las uniones realizadas hasta ahora que debemos abordar:

El DataFrame fusionado tiene los sufijos _x y _y agregados a los nombres de las columnas 'title'.
La columna en la que realizamos la unión tiene el mismo nombre en ambos DataFrames, 'author'.
Cuando se fusionan DataFrames en pandas, es importante asegurarse de que no hay dos columnas con el mismo nombre. En caso contrario, pandas añadirá automáticamente los sufijos _x y _y. Sin embargo, estos sufijos no son muy descriptivos. Para establecer mejores sufijos, pasa una tupla de sufijos al parámetro suffixes= en merge():

```py
both_pupils = first_pupil_df.merge(second_pupil_df,
                                   on='author',
                                   suffixes=('_1st_student', '_2nd_student')
                                  )
print(both_pupils)
```

      author             title_1st_student                   title_2nd_student
0  Steinbeck               Of Mice and Men                        East of Eden
1      Twain  The Adventures of Tom Sawyer  The Adventures of Huckleberry Finn
2  Hemingway       The Old Man and the Sea             For Whom the Bell Tolls


Ahora, los nombres de las columnas indican explícitamente el origen de las mismas. Siempre es una buena práctica usar nombres de columnas descriptivos como estos. Ten en cuenta que la primera cadena en la lista de sufijos se agrega al nombre de la columna del DataFrame izquierdo y la segunda cadena se agrega al DataFrame derecho.

En cuanto al segundo punto, no siempre es el caso de que las columnas que quieres fusionar tengan el mismo nombre. Puedes cambiar sus nombres para que coincidan antes de fusionar, pero esto puede causar confusión.

En cambio, la función merge() tiene los parámetros left_on= y right_on= que puedes usar en lugar de on= si las columnas tienen nombres diferentes. Para ilustrar cómo funciona, vamos a recrear los DataFrames para que uno tenga una columna 'authors' y el otro tenga una columna 'author':

```py

first_pupil_df = pd.DataFrame(
    {
        'authors': ['Alcott', 'Fitzgerald', 'Steinbeck', 'Twain', 'Hemingway'],
        'title': ['Little Women',
                  'The Great Gatsby',
                  'Of Mice and Men',
                  'The Adventures of Tom Sawyer',
                  'The Old Man and the Sea'
                 ],
    }
)
second_pupil_df = pd.DataFrame(
    {
        'author': ['Steinbeck', 'Twain', 'Hemingway', 'Salinger', 'Hawthorne'],
        'title': ['East of Eden',
                  'The Adventures of Huckleberry Finn',
                  'For Whom the Bell Tolls',
                  'The Catcher in the Rye',
                  'The Scarlett Letter'
                 ],
    }
)

both_pupils = first_pupil_df.merge(second_pupil_df,
                                   left_on='authors',
                                   right_on='author'
                                  )
print(both_pupils)
```

     authors                       title_x     author  \
0  Steinbeck               Of Mice and Men  Steinbeck   
1      Twain  The Adventures of Tom Sawyer      Twain   
2  Hemingway       The Old Man and the Sea  Hemingway   

                              title_y  
0                        East of Eden  
1  The Adventures of Huckleberry Finn  
2             For Whom the Bell Tolls

Esta vez, Pandas mostrará tanto 'authors' como 'author', pero estarán alineados y bien posicionados, para que puedas entender los resultados y cómo se relacionan entre sí.

# El método drop()
Ahora tenemos el resultado de la fusión interna, que contiene alguna información duplicada porque tanto 'author' como 'authors' se conservaron de los DataFrames originales, como vimos anteriormente.

Si queremos eliminar la información duplicada, podemos usar el método drop() con axis='columns' para señalar que queremos eliminar una columna en lugar de una fila:

```py
both_pupils = first_pupil_df.merge(second_pupil_df,
                                   left_on='authors',
                                   right_on='author'
                                  )
print(both_pupils.drop('author', axis='columns'))
```

     authors                       title_x                             title_y
0  Steinbeck               Of Mice and Men                        East of Eden
1      Twain  The Adventures of Tom Sawyer  The Adventures of Huckleberry Finn
2  Hemingway       The Old Man and the Sea             For Whom the Bell Tolls

# Resumen
- Fusión interna: solo conserva las filas con valores compartidos en la columna especificada de ambos DataFrames originales. Se hace con how='inner' (argumento por defecto).
- Fusión externa: conserva todos los valores de la columna especificada de ambos DataFrames originales, con los valores ausentes donde no haya coincidencia. Se hace con how='outer'.
- Fusión izquierda: conserva todos los valores del DataFrame izquierdo, y los valores del DataFrame derecho solo se conservan para los valores que coinciden con la columna especificada en el DataFrame izquierdo. Se hace con how='left'.
- Fusión derecha: conserva todos los valores del DataFrame derecho, y los valores del DataFrame izquierdo solo se conservan para los valores que coinciden con la columna especificada en el DataFrame derecho. Se hace con how='right'.
- Método drop(): elimina una columna de un DataFrame utilizando el parámetro axis='columns'.
Ahora es tu turno de probar tus habilidades de unión con un nuevo DataFrame en los ejercicios.